# Convex single-workflow demo

This notebook is the lightweight end-to-end reference workflow. Use it for exposition, debugging, and future onboarding. For thesis-scale aggregation, use the matching standalone runner:

```text
quantbayes/ball_dp/experiments/convex/run_thesis_experiment.py
```

The notebook deliberately keeps the workflow interactive; the script writes reproducible CSV/JSON summaries and figures.


# Convex tutorial: synthetic Ball-DP noisy ERM, finite-prior attacks, and bounds

This notebook is an end-to-end tutorial for the convex Gaussian-output part of the library. It uses synthetic embeddings so the workflow is easy to run and inspect. For real experiments, replace only the arrays `X_train`, `y_train`, `X_public`, and `y_public`; the finite-prior setup, attacks, diagnostics, and plots should stay the same.

The notebook demonstrates four things:

1. fit convex noisy-ERM releases under Ball-DP and standard-DP calibration;
2. build the canonical finite-prior replacement attack setup;
3. run exact finite-prior Gaussian-output attacks;
4. report utility, noise, exact-ID attack success, and direct finite-prior bounds.

## Mathematical setup

A record is an embedding-label pair $z=(x,y)$. We use the label-preserving metric

$$
d((x,y),(x',y')) =
\begin{cases}
\|x-x'\|_2, & y=y',\\
+\infty, & y \ne y'.
\end{cases}
$$

The canonical finite-prior replacement experiment fixes a public center $u$, removes that center from the private training set to form $D^-$, and constructs a finite support

$$
S = \{z_1,\ldots,z_m\} \subseteq B(u,r).
$$

The hidden record is sampled from a finite prior $\pi$ over $S$, usually the uniform prior. For each candidate $z_i$, the trained dataset is

$$
D_i = D^- \cup \{z_i\}.
$$

For convex Gaussian-output perturbation, the released object has the form

$$
\widetilde\theta = \theta(D_Z) + \xi,
\qquad
\xi \sim \mathcal N(0,\sigma^2 I).
$$

The exact finite-prior Bayes attack scores each candidate by

$$
\log \pi_i - \frac{\|\widetilde\theta - \theta(D_i)\|_2^2}{2\sigma^2}.
$$

The oblivious exact-identification baseline is

$$
\kappa = \max_i \pi_i,
$$

which equals $1/m$ for a uniform prior.

## Imports

The attack setup helpers are deliberately mechanism-agnostic. The convex-specific pieces are only used after the canonical trial object has been built.

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "quantbayes").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quantbayes.ball_dp import (
    ball_rero,
    fit_convex,
    make_finite_identification_prior,
)
from quantbayes.ball_dp.api import (
    attack_convex_finite_prior_trial,
    diagnose_convex_ball_output_finite_prior,
)
from quantbayes.ball_dp.attacks.finite_prior_setup import (
    CandidateSource,
    find_feasible_replacement_banks,
    make_replacement_trial,
    select_support_from_bank,
    target_positions_for_support,
)

plt.rcParams.update(
    {
        "figure.figsize": (7.4, 4.4),
        "figure.dpi": 130,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "legend.frameon": False,
    }
)

## Configuration

The defaults below are intentionally small. Increase `N_SAMPLES`, `M`, the number of release seeds, or the number of supports in real experiment scripts rather than in this tutorial.

In [ ]:
SEED = 0

# Synthetic embedding data.
N_SAMPLES = 800
N_FEATURES = 16
TEST_SIZE = 0.30
EMBEDDING_BOUND = 5.0

# Canonical finite-prior setup.
RADIUS = 1.75
STANDARD_RADIUS = 2.0 * EMBEDDING_BOUND
M = 6
SUPPORT_SELECTION = "farthest"  # "random", "nearest", or "farthest"

# Convex noisy ERM.
MODEL_FAMILY = "ridge_prototype"
EPSILON = 4.0
DELTA = 1e-6
LAM = 1e-2
MAX_ITER = 100
SOLVER = "lbfgs_fullbatch"
RIDGE_SENSITIVITY_MODE = "global"
ORDERS = tuple(list(range(2, 65)) + [80, 96, 128])

## Synthetic data

We generate a binary classification problem and scale all features into a public norm bound. This plays the role of precomputed embeddings in the paper-scale experiments.

In [ ]:
def make_synthetic_embeddings(*, seed: int):
    X, y = make_classification(
        n_samples=N_SAMPLES,
        n_features=N_FEATURES,
        n_informative=N_FEATURES,
        n_redundant=0,
        n_repeated=0,
        n_clusters_per_class=1,
        class_sep=2.8,
        flip_y=0.0,
        random_state=seed,
    )
    X = X.astype(np.float32)
    y = y.astype(np.int32)

    X_train, X_public, y_train, y_public = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=seed,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_public = scaler.transform(X_public).astype(np.float32)

    max_norm = max(
        float(np.linalg.norm(X_train, axis=1).max()),
        float(np.linalg.norm(X_public, axis=1).max()),
        1e-12,
    )
    scale = 0.95 * EMBEDDING_BOUND / max_norm
    return (
        (scale * X_train).astype(np.float32),
        y_train.astype(np.int32),
        (scale * X_public).astype(np.float32),
        y_public.astype(np.int32),
    )


X_train, y_train, X_public, y_public = make_synthetic_embeddings(seed=SEED)
NUM_CLASSES = int(len(np.unique(np.concatenate([y_train, y_public]))))
FEATURE_DIM = int(X_train.shape[1])

summary = pd.DataFrame(
    {
        "split": ["private train", "public/eval"],
        "n": [len(X_train), len(X_public)],
        "feature_dim": [FEATURE_DIM, FEATURE_DIM],
        "max_l2_norm": [
            float(np.linalg.norm(X_train, axis=1).max()),
            float(np.linalg.norm(X_public, axis=1).max()),
        ],
        "class_0": [int(np.sum(y_train == 0)), int(np.sum(y_public == 0))],
        "class_1": [int(np.sum(y_train == 1)), int(np.sum(y_public == 1))],
    }
)
display(summary)

## Build the canonical finite-prior support

The support construction is shared by convex and nonconvex attacks:

1. choose a center $u$ from the private training set;
2. remove it to form $D^-$;
3. find same-label public candidates within radius $r$;
4. choose a support $S \subseteq B(u,r)$ of size $m$.

For the convex tutorial we enumerate all target positions in $S$.

In [ ]:
public_source = CandidateSource("public", X_public, y_public)

bank = find_feasible_replacement_banks(
    X_train=X_train,
    y_train=y_train,
    candidate_sources=[public_source],
    radius=RADIUS,
    min_support_size=M,
    num_banks=1,
    seed=SEED,
    anchor_selection="large_bank",
    strict=True,
)[0]

support = select_support_from_bank(
    bank,
    m=M,
    selection=SUPPORT_SELECTION,
    seed=SEED,
    draw_index=0,
)

target_positions = target_positions_for_support(support, policy="all")

support_df = pd.DataFrame(
    {
        "support_position": np.arange(support.m),
        "source_id": list(support.source_ids),
        "label": support.y,
        "distance_to_center": support.distances_to_center,
        "prior_weight": support.weights,
    }
)

print("center:", support.center_source_id)
print("center label:", support.center_y)
print("support hash:", support.support_hash)
print("oblivious baseline kappa:", support.oblivious_kappa)
display(support_df)

## Visualize the support geometry

This projection is only for visualization. The actual support is selected in the full feature space.

In [ ]:
def plot_support_geometry(X_public, y_public, support):
    pca = PCA(n_components=2, random_state=SEED)
    all_points = np.concatenate(
        [X_public, support.center_x[None, :], support.X],
        axis=0,
    )
    Z = pca.fit_transform(all_points)
    Z_public = Z[: len(X_public)]
    z_center = Z[len(X_public)]
    Z_support = Z[len(X_public) + 1 :]

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for cls in sorted(np.unique(y_public).tolist()):
        mask = y_public == cls
        ax.scatter(
            Z_public[mask, 0],
            Z_public[mask, 1],
            s=18,
            alpha=0.20,
            label=f"public class {int(cls)}",
        )

    ax.scatter(
        [z_center[0]],
        [z_center[1]],
        s=170,
        marker="*",
        edgecolor="black",
        linewidth=0.8,
        label="center u",
        zorder=5,
    )
    ax.scatter(
        Z_support[:, 0],
        Z_support[:, 1],
        s=90,
        marker="o",
        edgecolor="black",
        linewidth=0.7,
        label="finite support S",
        zorder=4,
    )
    for i, z in enumerate(Z_support):
        ax.text(z[0], z[1], str(i), fontsize=9, ha="center", va="center")

    ax.set_title("Finite-prior support in a 2D PCA projection")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(loc="best")
    plt.show()


plot_support_geometry(X_public, y_public, support)

## Fit convex noisy-ERM releases for utility and generic bounds

We first fit Ball-DP and standard-DP noisy-ERM releases on the original synthetic training set. This is the utility/privacy part of the ERM cycle. The attack below refits releases on the replacement datasets $D^- \cup \{z_i\}$, because the finite-prior likelihood is defined over those hypotheses.

In [ ]:
def fit_noisy_convex_release(*, X, y, radius: float, seed: int):
    return fit_convex(
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.int32),
        X_eval=np.asarray(X_public, dtype=np.float32),
        y_eval=np.asarray(y_public, dtype=np.int32),
        model_family=MODEL_FAMILY,
        privacy="ball_dp",
        radius=float(radius),
        standard_radius=float(STANDARD_RADIUS),
        ridge_sensitivity_mode=RIDGE_SENSITIVITY_MODE,
        lam=float(LAM),
        epsilon=float(EPSILON),
        delta=float(DELTA),
        embedding_bound=float(EMBEDDING_BOUND),
        num_classes=int(NUM_CLASSES),
        orders=ORDERS,
        max_iter=int(MAX_ITER),
        solver=SOLVER,
        seed=int(seed),
    )


def first_epsilon(ledger) -> float:
    certs = getattr(ledger, "dp_certificates", [])
    return float("nan") if not certs else float(certs[0].epsilon)


utility_releases = {
    "Ball-DP": fit_noisy_convex_release(X=X_train, y=y_train, radius=RADIUS, seed=SEED),
    "Standard-DP": fit_noisy_convex_release(
        X=X_train, y=y_train, radius=STANDARD_RADIUS, seed=SEED
    ),
}

release_rows = []
for name, release in utility_releases.items():
    release_rows.append(
        {
            "mechanism": name,
            "accuracy": float(release.utility_metrics.get("accuracy", np.nan)),
            "sigma": float(release.privacy.ball.sigma),
            "release_radius": float(release.privacy.ball.radius),
            "delta_ball": float(release.sensitivity.delta_ball),
            "delta_standard": (
                np.nan
                if release.sensitivity.delta_std is None
                else float(release.sensitivity.delta_std)
            ),
            "epsilon_ball_view": first_epsilon(release.privacy.ball),
            "epsilon_standard_view": first_epsilon(release.privacy.standard),
        }
    )

release_df = pd.DataFrame(release_rows)
display(release_df)

## Generic exact-ID bounds for the finite support

For exact identification with the discrete $0/1$ loss, every threshold $\eta<1$ has

$$
\kappa(\eta) = \max_i \pi_i.
$$

The generic direct Gaussian bound uses the release sensitivity and noise scale. The instance diagnostics below are sharper because they use the actual finite candidate means.

In [ ]:
finite_prior = make_finite_identification_prior(
    support.X.reshape(support.m, -1),
    weights=support.weights,
)

bound_rows = []
for name, release in utility_releases.items():
    for mode in ["gaussian_direct", "rdp"]:
        try:
            report = ball_rero(
                release,
                prior=finite_prior,
                eta_grid=(0.5,),
                mode=mode,
            )
            point = report.points[0]
            bound_rows.append(
                {
                    "mechanism": name,
                    "bound_mode": mode,
                    "kappa": float(point.kappa),
                    "gamma_ball": float(point.gamma_ball),
                    "gamma_standard": (
                        np.nan
                        if point.gamma_standard is None
                        else float(point.gamma_standard)
                    ),
                }
            )
        except Exception as exc:
            bound_rows.append(
                {
                    "mechanism": name,
                    "bound_mode": mode,
                    "kappa": support.oblivious_kappa,
                    "gamma_ball": np.nan,
                    "gamma_standard": np.nan,
                    "error": str(exc),
                }
            )

generic_bound_df = pd.DataFrame(bound_rows)
display(generic_bound_df)

## Run the exact finite-prior Gaussian-output attack

For each target position $i \in \{1,\ldots,m\}$ we form the replacement dataset $D^- \cup \{z_i\}$, fit the noisy convex release, and attack over the same fixed support $S$.

This is the canonical protocol. The support is not rebuilt around the target.

In [ ]:
attack_rows = []
posterior_rows = []

for target_pos in target_positions:
    trial = make_replacement_trial(
        X_train=X_train,
        y_train=y_train,
        support=support,
        target_support_position=int(target_pos),
    )

    for mechanism, radius_value in [
        ("Ball-DP", RADIUS),
        ("Standard-DP", STANDARD_RADIUS),
    ]:
        release = fit_noisy_convex_release(
            X=trial.X_full,
            y=trial.y_full,
            radius=float(radius_value),
            seed=SEED,
        )

        attack = attack_convex_finite_prior_trial(
            release,
            trial,
            known_label=int(trial.support.center_y),
            eta_grid=(0.5,),
        )

        diagnostics = diagnose_convex_ball_output_finite_prior(
            release,
            trial.X_full,
            trial.y_full,
            target_index=int(trial.target_index),
            X_candidates=trial.support.X,
            y_candidates=trial.support.y,
            prior_weights=trial.support.weights,
            known_label=int(trial.support.center_y),
            center_features=trial.support.center_x,
            center_label=int(trial.support.center_y),
        )

        pred_idx = attack.diagnostics.get("predicted_prior_index")
        pred_sid = attack.diagnostics.get("predicted_source_id")

        attack_rows.append(
            {
                "mechanism": mechanism,
                "target_position": int(target_pos),
                "target_source_id": str(trial.target_source_id),
                "predicted_position": None if pred_idx is None else int(pred_idx),
                "predicted_source_id": pred_sid,
                "source_exact_id": float(
                    attack.metrics.get("source_exact_identification_success", np.nan)
                ),
                "feature_exact_id": float(
                    attack.metrics.get("exact_identification_success", np.nan)
                ),
                "prior_rank": float(attack.metrics.get("prior_rank", np.nan)),
                "posterior_true_probability": float(
                    attack.metrics.get("posterior_true_probability", np.nan)
                ),
                "posterior_top1_probability": float(
                    attack.metrics.get("posterior_top1_probability", np.nan)
                ),
                "posterior_effective_candidates": float(
                    attack.metrics.get("posterior_effective_candidates", np.nan)
                ),
                "instance_bound_finite_opt": float(
                    diagnostics.get("bound_direct_instance_finite_opt", np.nan)
                ),
                "instance_bound_center_max": float(
                    diagnostics.get("bound_direct_instance_center_max", np.nan)
                ),
                "model_pairwise_snr_median": float(
                    diagnostics.get("model_pairwise_snr_median", np.nan)
                ),
                "model_nn_snr_median": float(
                    diagnostics.get("model_nn_snr_median", np.nan)
                ),
                "sigma": float(release.privacy.ball.sigma),
                "accuracy": float(release.utility_metrics.get("accuracy", np.nan)),
            }
        )

        posterior = attack.diagnostics.get("candidate_posterior_probs")
        if posterior is not None:
            for j, p in enumerate(posterior):
                posterior_rows.append(
                    {
                        "mechanism": mechanism,
                        "target_position": int(target_pos),
                        "candidate_position": int(j),
                        "posterior_probability": float(p),
                        "is_true": bool(j == int(target_pos)),
                    }
                )

attack_df = pd.DataFrame(attack_rows)
posterior_df = pd.DataFrame(posterior_rows)

display(attack_df)

## Summarize the quantities of interest

The primary empirical quantity is exact-ID success. For the canonical source-ID accounting, success means the predicted support source ID matches the hidden target source ID.

The diagnostic bounds are not empirical accuracies. They upper-bound exact identification for the concrete finite Gaussian testing problem induced by the support.

In [ ]:
attack_summary = (
    attack_df.groupby("mechanism", dropna=False)
    .agg(
        n_trials=("source_exact_id", "count"),
        exact_id_mean=("source_exact_id", "mean"),
        feature_exact_id_mean=("feature_exact_id", "mean"),
        prior_rank_mean=("prior_rank", "mean"),
        posterior_true_mean=("posterior_true_probability", "mean"),
        posterior_top1_mean=("posterior_top1_probability", "mean"),
        posterior_effective_candidates_mean=("posterior_effective_candidates", "mean"),
        instance_bound_finite_opt_mean=("instance_bound_finite_opt", "mean"),
        instance_bound_center_max_mean=("instance_bound_center_max", "mean"),
        model_pairwise_snr_median_mean=("model_pairwise_snr_median", "mean"),
        sigma_mean=("sigma", "mean"),
        accuracy_mean=("accuracy", "mean"),
    )
    .reset_index()
)

attack_summary["baseline_kappa"] = float(support.oblivious_kappa)
display(attack_summary)

## Visualize utility, attacks, and bounds

A good tutorial figure should put the empirical exact-ID rate next to the oblivious baseline and the support-specific direct bound. The bound should be above the empirical attack rate up to sampling variability and numerical issues.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.0), constrained_layout=True)

# Utility/noise summary.
axes[0].bar(release_df["mechanism"], release_df["accuracy"], alpha=0.85)
axes[0].set_ylim(0.0, 1.05)
axes[0].set_ylabel("accuracy")
axes[0].set_title("Noisy ERM utility")

ax2 = axes[0].twinx()
ax2.plot(release_df["mechanism"], release_df["sigma"], marker="o", color="black")
ax2.set_ylabel("noise scale sigma")

# Empirical exact-ID and instance bound.
x = np.arange(len(attack_summary))
width = 0.26
axes[1].bar(
    x - width,
    attack_summary["exact_id_mean"],
    width=width,
    label="empirical exact-ID",
)
axes[1].bar(
    x,
    attack_summary["instance_bound_finite_opt_mean"],
    width=width,
    label="finite Gaussian bound",
)
axes[1].bar(
    x + width,
    attack_summary["baseline_kappa"],
    width=width,
    label="baseline kappa",
)
axes[1].set_xticks(x)
axes[1].set_xticklabels(attack_summary["mechanism"])
axes[1].set_ylim(0.0, 1.05)
axes[1].set_ylabel("probability")
axes[1].set_title("Exact-ID attack vs bounds")
axes[1].legend()

# Posterior confidence.
axes[2].bar(
    attack_summary["mechanism"],
    attack_summary["posterior_true_mean"],
    label="true posterior prob.",
)
axes[2].bar(
    attack_summary["mechanism"],
    attack_summary["posterior_top1_mean"],
    bottom=0,
    alpha=0.35,
    label="top-1 posterior prob.",
)
axes[2].axhline(
    support.oblivious_kappa,
    linestyle="--",
    color="gray",
    label="baseline",
)
axes[2].set_ylim(0.0, 1.05)
axes[2].set_ylabel("posterior probability")
axes[2].set_title("Posterior concentration")
axes[2].legend()

plt.show()

## Posterior over support candidates

The exact finite-prior attack computes a posterior over the fixed candidate support. The plot below shows one posterior distribution per mechanism for the first target position.

In [ ]:
if not posterior_df.empty:
    example_target = int(posterior_df["target_position"].iloc[0])
    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    for mechanism, sub in posterior_df[posterior_df["target_position"] == example_target].groupby("mechanism"):
        sub = sub.sort_values("candidate_position")
        ax.plot(
            sub["candidate_position"],
            sub["posterior_probability"],
            marker="o",
            label=mechanism,
        )
    ax.axhline(support.oblivious_kappa, linestyle="--", color="gray", label="uniform prior")
    ax.axvline(example_target, linestyle=":", color="black", label="true target")
    ax.set_xlabel("support candidate position")
    ax.set_ylabel("posterior probability")
    ax.set_title(f"Posterior over support for target position {example_target}")
    ax.legend()
    plt.show()
else:
    print("No posterior probabilities were recorded.")

## Reusing this notebook on real data

To move from this synthetic demo to real experiments, replace the synthetic data cell with your real arrays:

```python
X_train = ...
y_train = ...
X_public = ...
y_public = ...
```

Then keep the same canonical setup code:

```python
public_source = CandidateSource("public", X_public, y_public)
bank = find_feasible_replacement_banks(...)
support = select_support_from_bank(...)
trial = make_replacement_trial(...)
```

The attack setup is shared across convex and nonconvex experiments. Only the release training and attack scorer change.